# Qwen3-4B exp009 - đánh giá ABSA thực tế (Shopee)

**Mô hình:** `qwen3-4b-instruct-2507/prompt-cot/exp009`, dùng base `Qwen3-4B-Instruct-2507` lượng hoá 4-bit; id serving `qwen3_4b_exp009`.

**Chế độ:** chỉ suy luận - **không huấn luyện**, không cập nhật trọng số.

**Prompt-only:** exp009 **không có checkpoint/adapter**. Notebook chỉ nạp base model và prompt CoT 1-shot của exp009.

**Cấu hình:** nạp từ mã nguồn bằng `experiments.load("qwen3-4b-instruct-2507", "prompt-cot", "exp009")`. Không đọc `experiments/.../config.yaml` trên Drive; `configs/serving.yaml` chỉ dùng cho labels/aspects/default shell.

**Dữ liệu:** `data/raw/realworld/v0.1.0/shoppe_absa.csv` - 60 đánh giá son môi Shopee. Cột `text` được đưa **nguyên văn** vào serving runtime; không clean/normalize/segment bên ngoài runtime.

**Gói cài trên Colab:** `requirements-serving.txt` + `accelerate` + `bitsandbytes` (không `peft`, không `sentencepiece`, không cài `torch`).


In [ ]:
# --- BẢN CODE ĐÃ GHIM ---
REPO_URL = "https://github.com/TrieuKhac-dev/SentimentX"
REPO_BRANCH = "main"
REPO_SHA = "05906f647f7364636665236a773b9954e4b92ee5"

import importlib
import os
import pathlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def repo_root():
    if IN_COLAB:
        return pathlib.Path("/content/SentimentX")
    here = pathlib.Path.cwd()
    for candidate in [here] + list(here.parents):
        if (candidate / "configs" / "paths.yaml").is_file() and (candidate / "src" / "__init__.py").is_file():
            return candidate
    return here


def git(*args):
    done = subprocess.run(["git"] + [str(item) for item in args], capture_output=True, text=True)
    print("  git {} -> {}".format(" ".join(str(item) for item in args), done.returncode))
    if done.returncode != 0:
        print("    " + ((done.stdout or "") + (done.stderr or "")).strip()[-400:])
    return done.returncode


def head_sha(directory):
    done = subprocess.run(["git", "-C", str(directory), "rev-parse", "HEAD"],
                          capture_output=True, text=True)
    return done.stdout.strip() if done.returncode == 0 else ""


CODE_DIR = repo_root()

if IN_COLAB:
    from google.colab import drive
    if not pathlib.Path("/content/drive").is_mount():
        drive.mount("/content/drive")
    has_source = (CODE_DIR / "configs" / "paths.yaml").is_file() and (CODE_DIR / "src" / "__init__.py").is_file()
    if (not has_source) or head_sha(CODE_DIR) != REPO_SHA:
        if CODE_DIR.exists() and any(CODE_DIR.iterdir()) and not (CODE_DIR / ".git").is_dir():
            raise SystemExit("DỪNG: {} đã có nhưng không phải git repo. Xoá rồi chạy lại.".format(CODE_DIR))
        if not (CODE_DIR / ".git").is_dir():
            CODE_DIR.mkdir(parents=True, exist_ok=True)
            if git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(CODE_DIR)) != 0:
                raise SystemExit("DỪNG: không clone được {}".format(REPO_URL))
        git("-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", REPO_SHA)
        if git("-C", str(CODE_DIR), "checkout", "--detach", REPO_SHA) != 0:
            raise SystemExit("DỪNG: không checkout được commit {}".format(REPO_SHA))
    os.environ["HF_HOME"] = "/content/hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
    pathlib.Path("/content/hf_cache").mkdir(parents=True, exist_ok=True)
else:
    if not (CODE_DIR / "configs" / "paths.yaml").is_file():
        raise SystemExit("DỪNG: không tìm thấy gốc repo từ {}".format(pathlib.Path.cwd()))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
sys.path_importer_cache.pop(str(CODE_DIR), None)

from src.api import bootstrap, paths, runtime

info = bootstrap.prepare()
print("prepare(): colab={} drive={} stop={}".format(info["colab"], info["drive"], info["stop"]))
if IN_COLAB:
    drive_root = info["drive"] or pathlib.Path("/content/drive/MyDrive/SentimentX")
    os.environ["SENTIMENTX_DATA_ROOT"] = str(drive_root / "data")
    os.environ["SENTIMENTX_RESULTS_ROOT"] = str(drive_root / "experiments")
    print("Drive root :", drive_root)
else:
    os.environ.pop("SENTIMENTX_DATA_ROOT", None)
    os.environ.pop("SENTIMENTX_RESULTS_ROOT", None)

# Chỉ cài thêm gói cần cho đường suy luận này; không cài torch.
EXTRAS = ['accelerate', 'bitsandbytes']
if IN_COLAB:
    print("Cài thêm:", ", ".join(EXTRAS))
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", str(CODE_DIR / "requirements-serving.txt")] + EXTRAS,
        capture_output=True, text=True)
    print("  pip install ->", done.returncode)
    if done.returncode != 0:
        print(((done.stdout or "") + (done.stderr or "")).strip()[-800:])
        raise SystemExit("DỪNG: cài gói thất bại.")

print("CODE_DIR   :", CODE_DIR)
print("DATA_ROOT  :", paths.data_root())
print("RESULTS    :", paths.results_root())


In [ ]:
# Cấu hình ĐÃ HỢP NHẤT từ mã nguồn (không đọc config.yaml trên Drive).
import json
from dataclasses import replace
from pathlib import Path

from src.api import experiments, paths
from src.evaluation import runner
from serving.config import load_settings

MODEL_ID = "qwen3-4b-instruct-2507"
METHOD = "prompt-cot"
EXP_ID = "exp009"
SERVING_ID = "qwen3_4b_exp009"
MODEL_DIR_NAME = "Qwen3-4B-Instruct-2507"
REQUIRED_FILES = (
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "generation_config.json",
    "model.safetensors.index.json",
)
INDEX_FILE = "model.safetensors.index.json"


def missing_model_files(directory):
    directory = Path(directory)
    missing = [name for name in REQUIRED_FILES if not (directory / name).is_file()]
    index = directory / INDEX_FILE
    if index.is_file():
        try:
            payload = json.loads(index.read_text(encoding="utf-8"))
        except (OSError, ValueError) as exc:
            missing.append("{} ({})".format(INDEX_FILE, exc))
        else:
            for shard in sorted({str(value) for value in (payload.get("weight_map") or {}).values()}):
                if not (directory / shard).is_file():
                    missing.append(shard)
    return missing


merged = experiments.load(MODEL_ID, METHOD, EXP_ID)
config = merged["config"]

if config.get("approach") != "prompt":
    raise SystemExit("DỪNG: exp009 phải có approach='prompt', đang là {!r}.".format(config.get("approach")))

pre = config.get("preprocess") or {}
inf = config.get("inference") or {}
decoding = config.get("decoding") or {}
if "max_length" not in pre:
    raise SystemExit("DỪNG: merged config thiếu preprocess.max_length.")
if "quantization" not in inf or "dtype" not in inf:
    raise SystemExit("DỪNG: merged config thiếu inference.quantization/dtype.")

local_dir = paths.data_root() / "models" / MODEL_DIR_NAME
model_dir = None
tried = []
if local_dir.is_dir():
    missing = missing_model_files(local_dir)
    tried.append("{} (thiếu: {})".format(local_dir, ", ".join(missing) or "không"))
    if not missing:
        model_dir = local_dir
if model_dir is None:
    repo_id = str(config.get("checkpoint") or "Qwen/Qwen3-4B-Instruct-2507")
    try:
        from huggingface_hub import snapshot_download
        cached = Path(snapshot_download(repo_id=repo_id, local_files_only=True))
        missing = missing_model_files(cached)
        tried.append("{} (thiếu: {})".format(cached, ", ".join(missing) or "không"))
        if not missing:
            model_dir = cached
    except Exception as exc:
        tried.append("HF cache {}: {}".format(repo_id, exc))
if model_dir is None:
    print("DỪNG: không thấy base model Qwen3-4B-Instruct-2507 (exp009 là prompt-only, KHÔNG có checkpoint).")
    for item in tried:
        print("  -", item)
    print("  cần đủ config.json + tokenizer + generation_config.json + model.safetensors.index.json + shard trong", local_dir)
    raise SystemExit(1)


def resolve(value):
    path = Path(str(value))
    if path.is_absolute():
        return path
    candidate = Path(merged["dir"]) / path
    return candidate if candidate.exists() else paths.root() / path


prompt_path = resolve(config.get("prompt") or "")
examples_path = resolve(config.get("examples") or "") if config.get("examples") else None
system_path = resolve(config.get("system_prompt") or "") if config.get("system_prompt") else None
for label, path in (("prompt", prompt_path), ("examples", examples_path), ("system", system_path)):
    if path is None or not path.is_file():
        raise SystemExit("DỪNG: thiếu file {}: {}".format(label, path))

max_new_tokens = decoding.get("max_new_tokens")
if max_new_tokens is None:
    max_new_tokens = runner.DEFAULT_MAX_NEW_TOKENS
    print("Ghi chú: merged config không khai decoding.max_new_tokens; dùng mặc định runner =", max_new_tokens)
max_new_tokens = int(max_new_tokens)

settings = load_settings(models_override=[SERVING_ID])
shell = settings.models[SERVING_ID]
runtime_values = dict(shell.runtime)
runtime_values.update({
    "model_dir": str(model_dir),
    "quantization": str(inf["quantization"]),
    "dtype": str(inf["dtype"]),
    "max_length": int(pre["max_length"]),
    "max_new_tokens": max_new_tokens,
    "do_sample": str(decoding.get("mode") or "greedy").lower() == "sample",
    "prompt": str(prompt_path),
    "examples": str(examples_path) if examples_path else "",
    "system": str(system_path) if system_path else "",
})
for key in ("prompt_sha", "examples_sha", "system_sha"):
    runtime_values.pop(key, None)
settings.models[SERVING_ID] = replace(shell, runtime=runtime_values)
spec = settings.models[SERVING_ID]

print("Thí nghiệm :", "{}/{}/{}".format(MODEL_ID, METHOD, EXP_ID))
print("Đường chạy :", config.get("approach"), "(prompt-only; KHÔNG có checkpoint)")
print("Model dir  :", model_dir)
print("Quant/dtype:", runtime_values["quantization"], "/", runtime_values["dtype"])
print("max_length :", runtime_values["max_length"], "| max_new_tokens:", runtime_values["max_new_tokens"])
print("Prompt     :", prompt_path)
print("Examples   :", examples_path)
print("System     :", system_path)
print("Aspects    :", settings.labels.get("aspects"))
print("Lớp config :")
for label, path in merged["layers"]:
    print("  {:<12} {}".format(label, path))


In [ ]:
# Dữ liệu thô + suy luận bằng serving runtime (đúng đường serving, không gọi encoder_run).
import hashlib
import time
from datetime import datetime, timezone

import pandas as pd

from src.api import paths, runtime as workflow_runtime
from serving.inference.llm import LLMRuntime
from src.evaluation import scorers

ASPECTS = [str(name) for name in settings.labels["aspects"]]
TASK = {"label_space": "binary", "neutral_policy": "drop", "not_mentioned": "separate"}
LABELS = {0: "not_mentioned", 1: "positive", 2: "negative"}
CHUNK = 1
INCLUDE_PROBS = False

data_path = paths.data_root() / "raw" / "realworld" / "v0.1.0" / "shoppe_absa.csv"
if not data_path.is_file():
    data_path = paths.root() / "data" / "raw" / "realworld" / "v0.1.0" / "shoppe_absa.csv"
if not data_path.is_file():
    raise SystemExit("DỪNG: không thấy data/raw/realworld/v0.1.0/shoppe_absa.csv ở data root hoặc repo.")

frame = pd.read_csv(data_path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
if len(frame) != 60:
    raise SystemExit("DỪNG: shoppe_absa.csv phải có 60 dòng, đang có {}.".format(len(frame)))
missing_cols = [name for name in ["text"] + ASPECTS if name not in frame.columns]
if missing_cols:
    raise SystemExit("DỪNG: CSV thiếu cột: {}".format(", ".join(missing_cols)))

texts = frame["text"].tolist()
gold_rows = []
for index in range(len(frame)):
    row = {}
    for aspect in ASPECTS:
        value = str(frame.iloc[index][aspect]).strip()
        if value not in ("0", "1", "2"):
            raise SystemExit("DỪNG: nhãn {!r} của cột {} dòng {} không thuộc 0/1/2.".format(value, aspect, index))
        row[aspect] = int(value)
    gold_rows.append(row)

row_ids = frame["row_id"].tolist() if "row_id" in frame.columns else list(range(len(frame)))
urls = frame["url"].tolist() if "url" in frame.columns else [""] * len(frame)

csv_sha = hashlib.sha256(data_path.read_bytes()).hexdigest()
run_id = hashlib.sha256((str(spec.version) + csv_sha).encode("utf-8")).hexdigest()[:8]
out_dir = paths.results_root() / "realworld" / SERVING_ID / run_id
out_dir.mkdir(parents=True, exist_ok=True)


def predict_chunked(runtime_obj, texts_in, chunk, include_probs):
    predictions, errors = [], []
    for start in range(0, len(texts_in), chunk):
        batch = texts_in[start:start + chunk]
        try:
            results = runtime_obj.predict(batch, include_probs=include_probs)
            if not isinstance(results, list) or len(results) != len(batch):
                raise RuntimeError(
                    "số kết quả {} khác số dòng {}".format(
                        len(results) if isinstance(results, list) else "?", len(batch)))
            predictions.extend(results)
            continue
        except Exception as exc:
            print("  khối {}-{} lỗi ({}); thử lại từng dòng.".format(
                start, start + len(batch) - 1, exc))
        for offset, text in enumerate(batch):
            index = start + offset
            try:
                one = runtime_obj.predict([text], include_probs=include_probs)
                if not one:
                    raise RuntimeError("không có kết quả")
                predictions.append(one[0])
            except Exception as row_exc:
                predictions.append(None)
                errors.append({
                    "index": index,
                    "row_id": row_ids[index],
                    "error": "{}: {}".format(type(row_exc).__name__, row_exc),
                })
    return predictions, errors


started_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
started = time.perf_counter()
runtime_obj = LLMRuntime()
try:
    runtime_obj.load(spec, settings)
    predictions, errors = predict_chunked(runtime_obj, texts, CHUNK, INCLUDE_PROBS)
finally:
    runtime_obj.unload()
elapsed = round(time.perf_counter() - started, 3)

pred_rows = []
for prediction in predictions:
    if prediction is None:
        pred_rows.append(None)
        continue
    pred_rows.append({
        aspect: (None if prediction.get(aspect) is None else prediction[aspect].code)
        for aspect in ASPECTS
    })

print("Đã dự đoán {} dòng | lỗi runtime: {}".format(len(predictions), len(errors)))
print("Hash8      :", run_id)
print("Thư mục    :", out_dir)
print("Dữ liệu    :", data_path)


In [ ]:
# Chấm điểm bằng src.evaluation.scorers.write + ghi predictions.csv, run_meta.json.
import json
from datetime import datetime, timezone

import pandas as pd

from src.evaluation import scorers

prediction_columns = {
    "row_id": row_ids,
    "text": texts,
    "url": urls,
}
for aspect in ASPECTS:
    prediction_columns["gold_" + aspect] = [row[aspect] for row in gold_rows]
    prediction_columns["pred_" + aspect] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].code
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_label"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].label
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_confidence"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].confidence
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_quote"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].quote
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_reason"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].reason
        for prediction in predictions
    ]

predictions_path = out_dir / "predictions.csv"
pd.DataFrame(prediction_columns).to_csv(predictions_path, index=False, encoding="utf-8-sig")

run_info = {
    "model": SERVING_ID,
    "model_version": spec.version,
    "run_id": run_id,
    "dataset": "shoppe_absa",
    "data_path": str(data_path),
    "csv_sha256": csv_sha,
    "aspects": ASPECTS,
    "task": TASK,
    "labels": LABELS,
    "chunk": CHUNK,
    "include_probs": INCLUDE_PROBS,
    "n_rows": len(texts),
    "n_errors": len(errors),
    "runtime_errors": errors,
    "config_dir": str(merged.get("dir")),
    "config_layers": list(merged.get("layers") or []),
    "config_sources": {
        key: merged.get("sources", {}).get(key)
        for key in ("approach", "checkpoint", "preprocess.max_length", "preprocess.segmenter",
                    "inference.dtype", "inference.quantization", "data.dataset", "data.version")
        if merged.get("sources", {}).get(key)
    },
    "runtime_spec": {
        key: spec.runtime.get(key)
        for key in ("checkpoint", "base", "base_revision", "max_length", "segmenter", "dtype",
                    "model_dir", "quantization", "max_new_tokens", "prompt", "examples", "system",
                    "local_files_only")
        if spec.runtime.get(key) is not None
    },
    "env": workflow_runtime.env_name(),
    "data_root": str(paths.data_root()),
    "results_root": str(paths.results_root()),
    "started_at": started_at,
    "seconds": elapsed,
}

samples = scorers.Samples.build(
    ASPECTS,
    gold_rows,
    pred_rows,
    task=dict(TASK),
    labels=dict(LABELS),
    sample_ids=texts,
    meta=dict(run_info),
)

names = list(config.get("scores") or scorers.available())
paper = list(config.get("scores_paper") or [])
save = config.get("save") or {}
written = scorers.write(
    out_dir,
    samples,
    names=names,
    paper=paper or None,
    save_confusion=bool(save.get("confusion", True)),
    save_plots=False,
    extra=dict(run_info),
)

metrics_path = out_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
scores = metrics.get("scores") or {}
accuracy = scores.get("accuracy") or {}
detection = scores.get("aspect_detection") or {}
accuracy_macro = accuracy.get("macro")
accuracy_micro = accuracy.get("micro")
detection_macro = (detection.get("macro") or {}).get("f1")
detection_micro = (detection.get("micro") or {}).get("f1")

files = {name: str(path) for name, path in written.items()}
files["predictions.csv"] = str(predictions_path)
run_meta = dict(run_info)
run_meta.update({
    "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "out_dir": str(out_dir),
    "files": files,
    "metrics": {
        "accuracy_macro": accuracy_macro,
        "accuracy_micro": accuracy_micro,
        "accuracy_cells": accuracy.get("cells"),
        "detection_f1_macro": detection_macro,
        "detection_f1_micro": detection_micro,
    },
})
run_meta_path = out_dir / "run_meta.json"
run_meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

print("=" * 72)
print("Mô hình       :", SERVING_ID)
print("Phiên bản     :", spec.version)
print("Hash8         :", run_id)
print("Thư mục       :", out_dir)
print("Accuracy      : macro={} | micro={} | số ô={}".format(
    accuracy_macro, accuracy_micro, accuracy.get("cells")))
print("Detection F1  : macro={} | micro={}".format(detection_macro, detection_micro))
print("Lỗi runtime   :", len(errors))
print("Thời gian     :", elapsed, "giây")
print("File          :")
for name in ("predictions.csv", "metrics.json", "metrics.csv", "mispredictions.csv", "run_meta.json"):
    print("  {} -> {}".format(name, out_dir / name))
print("=" * 72)
